# Day 1.2 — Configuring Model Behaviour
Instructions and settings *steer* a probabilistic generator; they do not program it.
This section shows what a system message and `temperature` change, and then shows that an
instruction is not a contract.

### Step 1 — Same question, with and without standing instructions

The **system** message is the job description; the **user** message is today's task.
Compare the reply when we demand a fixed three-section format.

In [ ]:
question = {"role": "user", "content": "Explain recursion."}

free = chat([question])
constrained = chat([
    {"role": "system", "content": "Answer in exactly three sections labelled Definition, Example, When to avoid. One sentence each."},
    question,
])
print("WITHOUT system message:\n", free["content"][:300])
print("\nWITH system message:\n", constrained["content"])

### Step 2 — `temperature`

`temperature` controls how freely the model samples the next token. `0` takes the most likely
continuation every time (use it for tools, extraction, pipelines); higher values trade
repeatability for variety (brainstorming). It changes *how adventurous* the reply is, not *what the job is*.

In [ ]:
for temperature in (0.0, 1.0):
    reply = chat([{"role": "user", "content": "Give me one creative name for a study-group app."}], temperature=temperature)
    print(f"temperature={temperature}: {reply['content'][:90]}")
print("\nThe mock ignores temperature. Live, run this cell twice: temperature=0 repeats, temperature=1 varies.")

### Step 3 — Break it: an instruction is not a contract

Ask nicely for a dictionary. The reply *looks* right to a human and still cannot be parsed by
Python. This is the problem the next section solves.

In [ ]:
reply = chat([{"role": "user", "content": "Give me a dictionary with keys task, due and priority for 'write the lab report by Friday'."}])
print("REPLY:", reply["content"])
try:
    json.loads(reply["content"])
    print("Parsed as JSON - lucky this time; it is not guaranteed.")
except json.JSONDecodeError as error:
    print("\njson.loads FAILED:", error)
    print("Looks fine to a person, useless to a program.")

### Checkpoint

**1. A teammate says: 'I told it to always return JSON in the system prompt, so parsing is safe.' What is wrong?**

<details><summary>Show answer</summary>

A prompt raises the probability of JSON; it does not guarantee it. Only a schema-enforced request plus validation in code makes parsing safe. Section 1.3 does exactly that.

</details>

**2. When would you deliberately use `temperature=1`?**

<details><summary>Show answer</summary>

When variety is the goal and no code depends on the exact output: brainstorming names, drafting alternatives. Never for tool selection or data extraction.

</details>

### Recap

- **Limitation seen:** wording and settings steer output but do not guarantee its shape.
- **Layer added:** a system message for standing rules; `temperature` chosen per job.
- **Evidence:** the 'dictionary' reply looked right and still broke `json.loads`.